# Generación de figuras — Sección 8 (Resultados)

Notebook autocontenido para Colab nuevo. Ejecutar de arriba abajo: las celdas de setup montan Drive, clonan el repo y preparan el dataset; el resto genera todas las figuras.

| Figura / Entregable | Archivo | GPU |
|---|---|---|
| Matrices de confusión E1b | `matrices_confusion_e1b_comparativa.png` | No |
| Barras de métricas E1b | `barras_metricas_e1b.png` | No |
| Curvas ROC en paneles (PNGs Drive) | `roc_curves_e1b_paneles.png` | No |
| Predictions por modelo | `{modelo}_e1b_predictions.pt` (×3) | Sí |
| CSV desglose por GAN | `desglose_e1b_por_arquitectura.csv` | Sí |
| Curvas ROC superpuestas | `roc_curves_e1b_superpuestas.png` | Sí\* |
| Heatmap AUC por generador | `heatmap_auc_por_generador.png` | Sí\* |

\* Recargan desde archivos guardados si la sesión se reinicia.

---
## Setup — Colab desde cero

Ejecutar una sola vez al inicio. No se necesita progan_train, wandb ni los splits de entrenamiento.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/manuelpalasanchez/tfm_deteccion.git /content/tfm_deteccion
!pip install -q -r /content/tfm_deteccion/requirements.txt
print('Setup completado.')

In [ ]:
import zipfile, shutil
from pathlib import Path

zip_src = Path('/content/drive/MyDrive/cnndetection-datasets/CNN_synth_testset.zip')
dst = Path('/content/CNN_synth_testset')
dst.mkdir(exist_ok=True)
tmp = Path('/content/_synth_tmp')

print(f'Extrayendo {zip_src.name}...')
with zipfile.ZipFile(str(zip_src)) as zf:
    zf.extractall(str(tmp))

# Detectar si el zip extrajo a un subfolder o directo al root
extracted = list(tmp.iterdir())
scan_dir = extracted[0] if len(extracted) == 1 and extracted[0].is_dir() else tmp

# Renombrar cada arch a {arch}_test/; excluir progan (es E1 in-dist, no E1b)
for arch_dir in sorted(scan_dir.iterdir()):
    if not arch_dir.is_dir():
        continue
    if arch_dir.name == 'progan':
        print(f'  progan -> OMITIDO (E1 in-dist, no necesario para figuras)')
        continue
    target = dst / f'{arch_dir.name}_test'
    shutil.move(str(arch_dir), str(target))
    print(f'  {arch_dir.name} -> {target.name}')

shutil.rmtree(str(tmp), ignore_errors=True)
print('\nCNN_synth_testset listo:')
!ls /content/CNN_synth_testset/

---
## Imports y configuración

In [ ]:
import csv
import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import seaborn as sns

try:
    import google.colab
    IN_COLAB = True
    BASE_DIR = Path('/content/tfm_deteccion')
except ImportError:
    IN_COLAB = False
    cwd = Path.cwd()
    BASE_DIR = cwd.parent if cwd.name == 'notebooks' else cwd

sys.path.insert(0, str(BASE_DIR))

FIGURAS_DIR = BASE_DIR / 'figuras'
FIGURAS_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    'font.family': 'serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print(f'BASE_DIR   : {BASE_DIR}')
print(f'FIGURAS_DIR: {FIGURAS_DIR}')
print(f'IN_COLAB   : {IN_COLAB}')

In [ ]:
import json

DRIVE_ROOT = Path('/content/drive/MyDrive/tfm-checkpoints')

modelo_folders = {
    'ResNet-50':           'resnet50_20260508_115724',
    'ViT-B/16':            'vit_20260509_122527',
    'UniversalFakeDetect': 'universalfakedetect_20260509_153838',
}
MODELOS = list(modelo_folders.keys())
COLORES = {'ResNet-50': '#1976D2', 'ViT-B/16': '#388E3C', 'UniversalFakeDetect': '#F57C00'}

resultados   = {}
matrices_e1b = {}

for nombre, folder in modelo_folders.items():
    mpath = DRIVE_ROOT / folder / 'metrics.json'
    with mpath.open() as f:
        m = json.load(f)
    e1b = m['e1b']
    resultados[nombre] = {
        'e1':  {'auc': m['e1']['auc_roc'], 'ap': m['e1']['average_precision'], 'acc': m['e1']['accuracy']},
        'e1b': {'auc': e1b['auc_roc'], 'ap': e1b['average_precision'], 'acc': e1b['accuracy'], 'n': e1b.get('n_samples', 0)},
    }
    cm_val = e1b.get('confusion_matrix')
    if cm_val:
        matrices_e1b[nombre] = cm_val

print('Metricas cargadas desde Drive:')
for nombre, vals in resultados.items():
    auc = vals['e1b']['auc']
    ap  = vals['e1b']['ap']
    acc = vals['e1b']['acc']
    n   = vals['e1b']['n']
    print(f'  {nombre}: E1b AUC={auc:.4f}  AP={ap:.4f}  Acc={acc:.4f}  n={n}')
    if nombre in matrices_e1b:
        print(f'    CM={matrices_e1b[nombre]}')


---
## Figura 1 — Matrices de confusión E1b comparativa

Tres paneles horizontales, uno por modelo. No requiere GPU ni datos adicionales.

In [ ]:
if not matrices_e1b:
    print('AVISO: confusion_matrix no en metrics.json. Re-evalua con metrics.py actualizado.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    tick_labels = ['Real', 'Fake']

    for ax, modelo in zip(axes, MODELOS):
        cm = np.array(matrices_e1b[modelo])
        cm_norm = cm.astype(float) / cm.max()
        sns.heatmap(cm, annot=False, cmap='Blues',
                    xticklabels=tick_labels, yticklabels=tick_labels,
                    ax=ax, cbar=False, linewidths=0.5, linecolor='white')
        for i in range(2):
            for j in range(2):
                val = cm[i, j]
                val_str = f'{val:,}'.replace(',', '.')
                text_color = 'white' if cm_norm[i, j] > 0.5 else 'black'
                ax.text(j + 0.5, i + 0.5, val_str,
                        ha='center', va='center', fontsize=13, fontweight='bold', color=text_color)
        ax.set_title(modelo, fontsize=13, pad=10)
        ax.set_xlabel('Predicho', fontsize=11)
        ax.set_ylabel('Clase real', fontsize=11)
        ax.tick_params(axis='both', labelsize=10)

    n_total = resultados[MODELOS[0]]['e1b']['n']
    n_str = f'{n_total:,}'.replace(',', '.')
    plt.suptitle(
        f'Matrices de confusion — Evaluacion E1b (cross-architecture, {n_str} imagenes)',
        fontsize=14, y=1.03,
    )
    plt.tight_layout()

    out_path = FIGURAS_DIR / 'matrices_confusion_e1b_comparativa.png'
    plt.savefig(str(out_path), dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Guardado: {out_path}')


---
## Figura 2 — Barras de métricas E1b

Barras agrupadas (AUC-ROC, AP, Accuracy) por modelo. Eje Y desde 0.70. No requiere GPU.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

metricas_keys   = ['auc',     'ap',    'acc']
metricas_labels = ['AUC-ROC', 'AP',    'Accuracy']
bar_colors      = ['#1976D2', '#388E3C', '#F57C00']
width   = 0.25
x       = np.arange(len(MODELOS))
offsets = [-width, 0.0, width]

for key, label, color, offset in zip(metricas_keys, metricas_labels, bar_colors, offsets):
    valores = [resultados[m]['e1b'][key] for m in MODELOS]
    bars = ax.bar(x + offset, valores, width,
                  label=label, color=color, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, valores):
        fmt_str = f'{val * 100:.2f}%' if key == 'acc' else f'{val:.4f}'
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.002,
                fmt_str, ha='center', va='bottom', fontsize=8.5)

ax.set_ylim(0.70, 1.03)
ax.set_xticks(x)
ax.set_xticklabels(MODELOS, fontsize=11)
ax.set_ylabel('Valor de la metrica', fontsize=12)
ax.set_title('Comparativa de metricas — Evaluacion E1b (cross-architecture)', fontsize=13)
ax.legend(fontsize=11, loc='lower right')
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)

plt.tight_layout()

out_path = FIGURAS_DIR / 'barras_metricas_e1b.png'
plt.savefig(str(out_path), dpi=300, bbox_inches='tight')
plt.show()
print(f'Guardado: {out_path}')

---
## Figura 3B — Curvas ROC E1b en paneles (sin GPU)

Compone los tres `roc_curve_e1b.png` ya generados en una sola figura. Requiere Drive montado.

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/tfm-checkpoints'

roc_pngs = {
    'ResNet-50':           Path(DRIVE_ROOT) / 'resnet50_20260508_115724'            / 'roc_curve_e1b.png',
    'ViT-B/16':            Path(DRIVE_ROOT) / 'vit_20260509_122527'                 / 'roc_curve_e1b.png',
    'UniversalFakeDetect': Path(DRIVE_ROOT) / 'universalfakedetect_20260509_153838' / 'roc_curve_e1b.png',
}

for modelo, ruta in roc_pngs.items():
    estado = 'OK' if ruta.exists() else 'NO ENCONTRADO'
    print(f'  {modelo}: {estado}')
    print(f'    {ruta}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 7))

for ax, (modelo, png_path) in zip(axes, roc_pngs.items()):
    img = mpimg.imread(str(png_path))
    ax.imshow(img)
    ax.set_title(modelo, fontsize=14, pad=10)
    ax.axis('off')

plt.suptitle('Curvas ROC — Evaluacion E1b (cross-architecture)', fontsize=15, y=1.01)
plt.tight_layout()

out_path = FIGURAS_DIR / 'roc_curves_e1b_paneles.png'
plt.savefig(str(out_path), dpi=300, bbox_inches='tight')
plt.show()
print(f'Guardado: {out_path}')

---
## Bloque GPU — Inferencia, desglose y visualizaciones

**Requiere**: GPU disponible, dataset E1b en `/content/CNN_synth_testset/` (preparado en cell-setup-b), checkpoints en Drive.

Un único pase de inferencia por modelo produce:
- `{modelo}_e1b_predictions.pt` — scores y targets completos de E1b
- `desglose_e1b_por_arquitectura.csv` — AUC / AP / Acc por modelo × arquitectura GAN
- Datos en memoria para Fig 3A y heatmap sin reejecutar inferencia

Tiempo estimado en T4: ~5-10 min por modelo (30.600 imágenes × 3 modelos).

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/tfm-checkpoints'
E1B_ROOT   = '/content/CNN_synth_testset'

modelos_3a = {
    'ResNet-50': {
        'config':     BASE_DIR / 'configs/resnet50.yaml',
        'checkpoint': Path(DRIVE_ROOT) / 'resnet50_20260508_115724'            / 'checkpoint_best.pth',
        'module':     'models.resnet50',
        'csv_key':    'resnet50',
        'color':      '#1976D2',
    },
    'ViT-B/16': {
        'config':     BASE_DIR / 'configs/vit.yaml',
        'checkpoint': Path(DRIVE_ROOT) / 'vit_20260509_122527'                 / 'checkpoint_best.pth',
        'module':     'models.vit',
        'csv_key':    'vit',
        'color':      '#388E3C',
    },
    'UniversalFakeDetect': {
        'config':     BASE_DIR / 'configs/universalfakedetect.yaml',
        'checkpoint': Path(DRIVE_ROOT) / 'universalfakedetect_20260509_153838' / 'checkpoint_best.pth',
        'module':     'models.universalfakedetect',
        'csv_key':    'ufd',
        'color':      '#F57C00',
    },
}

print('Verificando rutas:')
for nombre, info in modelos_3a.items():
    ckpt_ok = 'OK' if info['checkpoint'].exists() else 'NO ENCONTRADO'
    print(f'  {nombre}: {ckpt_ok}')
e1b_ok = 'OK' if Path(E1B_ROOT).exists() else 'NO ENCONTRADO'
n_arqs = len(list(Path(E1B_ROOT).glob('*_test'))) if Path(E1B_ROOT).exists() else 0
print(f'  E1B_ROOT: {e1b_ok}  ({n_arqs} arquitecturas GAN)')

In [ ]:
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import roc_curve, auc as sklearn_auc
from tqdm import tqdm

import data.cnndetection_dataset  # registra el dataset en el factory
from models import model_registry
from utils.config import load_config
from data.cnndetection_dataset import CNNDetectionDataset
from data.transforms import get_eval_transforms
from evaluation.metrics import compute_metrics

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cpu':
    print('AVISO: sin GPU disponible. La inferencia sera muy lenta.')

In [ ]:
# Un pase de inferencia por modelo -> roc_data + desglose_data + predictions .pt
desglose_data = {}  # arch -> {n_samples, resnet50_auc, ..., vit_auc, ..., ufd_auc, ...}
roc_data      = {}  # nombre_modelo -> {fpr, tpr, auc, color}

for nombre, info in modelos_3a.items():
    csv_key = info['csv_key']
    print('\n' + '='*55)
    print(f'  Modelo: {nombre}  (csv_key={csv_key})')
    print('='*55)

    # --- Cargar modelo ---
    cfg = load_config(info['config'])
    importlib.import_module(info['module'])
    model = model_registry.build(cfg.model.name, **vars(cfg.model.kwargs))
    ckpt = torch.load(str(info['checkpoint']), map_location='cpu')
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(DEVICE).eval()
    best_auc = ckpt.get('best_auc', float('nan'))
    print(f'  Checkpoint cargado (val best_auc={best_auc:.4f})')

    # --- Dataset completo E1b ---
    dataset = CNNDetectionDataset(root=E1B_ROOT, split='test', transform=get_eval_transforms())
    loader  = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
    print(f'  Dataset E1b: {len(dataset)} imagenes')

    # --- Inferencia ---
    all_scores, all_targets, all_generators = [], [], []
    with torch.no_grad():
        for images, labels, generators in tqdm(loader, desc='  Inferencia'):
            images = model.preprocess(images.to(DEVICE))
            logits = model(images).squeeze(1)
            scores = torch.sigmoid(logits).cpu().numpy()
            all_scores.extend(scores.tolist())
            all_targets.extend(labels.numpy().tolist())
            all_generators.extend(generators)

    y_true  = np.array(all_targets)
    y_score = np.array(all_scores)
    gens    = np.array(all_generators)

    # --- Guardar predictions .pt ---
    pt_path = FIGURAS_DIR / f'{csv_key}_e1b_predictions.pt'
    torch.save({
        'y_true':     torch.tensor(y_true,  dtype=torch.float32),
        'y_score':    torch.tensor(y_score, dtype=torch.float32),
        'generators': list(gens),
    }, str(pt_path))
    print(f'  Predictions guardadas: {pt_path}')

    # --- Curva ROC global E1b ---
    fpr, tpr, _ = roc_curve(y_true, y_score)
    auc_global  = sklearn_auc(fpr, tpr)
    roc_data[nombre] = {'fpr': fpr, 'tpr': tpr, 'auc': auc_global, 'color': info['color']}
    print(f'  AUC-ROC global E1b: {auc_global:.4f}')

    # --- Metricas por arquitectura GAN ---
    arqs = sorted(set(gens))
    arqs_str = ', '.join(arqs)
    print(f'  Arquitecturas ({len(arqs)}): {arqs_str}')
    print('  ' + '-'*55)
    for arch in arqs:
        mask = gens == arch
        m = compute_metrics(y_true[mask], y_score[mask])
        if arch not in desglose_data:
            desglose_data[arch] = {'n_samples': int(mask.sum())}
        desglose_data[arch][f'{csv_key}_auc'] = m['auc_roc']
        desglose_data[arch][f'{csv_key}_ap']  = m['average_precision']
        desglose_data[arch][f'{csv_key}_acc'] = m['accuracy']
        auc_v = m['auc_roc']
        ap_v  = m['average_precision']
        acc_v = m['accuracy']
        print(f'  {arch:25s}  AUC={auc_v:.4f}  AP={ap_v:.4f}  Acc={acc_v:.4f}')

    # --- Liberar GPU ---
    del model
    torch.cuda.empty_cache()

print(f'\nInferencia completada: {list(roc_data.keys())}')

In [ ]:
# Guardar CSV y mostrar tabla resumen
CSV_COLS = [
    'arquitectura_gan', 'n_samples',
    'resnet50_auc', 'resnet50_ap', 'resnet50_acc',
    'vit_auc',      'vit_ap',      'vit_acc',
    'ufd_auc',      'ufd_ap',      'ufd_acc',
]

csv_path = FIGURAS_DIR / 'desglose_e1b_por_arquitectura.csv'
with csv_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=CSV_COLS, extrasaction='ignore')
    writer.writeheader()
    for arch in sorted(desglose_data.keys()):
        row = {'arquitectura_gan': arch, **desglose_data[arch]}
        writer.writerow(row)

print(f'CSV guardado: {csv_path}\n')
print('Arquitectura GAN           N      ResNet AUC   ViT AUC   UFD AUC')
print('-' * 65)
for arch in sorted(desglose_data.keys(),
                   key=lambda a: -desglose_data[a].get('resnet50_auc', 0)):
    d = desglose_data[arch]
    n     = d['n_samples']
    r_auc = d.get('resnet50_auc', float('nan'))
    v_auc = d.get('vit_auc',      float('nan'))
    u_auc = d.get('ufd_auc',      float('nan'))
    print(f'{arch:25s}  {n:>6}  {r_auc:>10.4f}  {v_auc:>9.4f}  {u_auc:>9.4f}')

---
## Figura 3A — Curvas ROC E1b superpuestas

Tres curvas en un solo gráfico. Usa `roc_data` de la inferencia anterior; si la sesión se reinició, recarga desde los `.pt`.

In [ ]:
# Recargar desde .pt si roc_data no esta en memoria (sesion reiniciada)
if 'roc_data' not in globals() or not roc_data:
    print('roc_data no en memoria, reconstruyendo desde .pt...')
    if 'modelos_3a' not in globals():
        raise RuntimeError('Ejecutar primero la celda de configuracion de modelos (cell-12).')
    try:
        import torch
        from sklearn.metrics import roc_curve, auc as sklearn_auc
    except ImportError:
        pass
    roc_data = {}
    for nombre, info in modelos_3a.items():
        csv_key = info['csv_key']
        pt_path = FIGURAS_DIR / f'{csv_key}_e1b_predictions.pt'
        saved   = torch.load(str(pt_path), map_location='cpu')
        y_true  = saved['y_true'].numpy()
        y_score = saved['y_score'].numpy()
        fpr, tpr, _ = roc_curve(y_true, y_score)
        roc_auc = sklearn_auc(fpr, tpr)
        roc_data[nombre] = {'fpr': fpr, 'tpr': tpr, 'auc': roc_auc, 'color': info['color']}
        print(f'  {nombre}: AUC={roc_auc:.4f}')

fig, ax = plt.subplots(figsize=(8, 8))
for nombre, rd in roc_data.items():
    roc_color = rd['color']
    roc_auc   = rd['auc']
    ax.plot(rd['fpr'], rd['tpr'], color=roc_color, lw=2.5,
            label=f'{nombre} (AUC = {roc_auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Clasificador aleatorio')
ax.set_xlabel('Tasa de Falsos Positivos', fontsize=13)
ax.set_ylabel('Tasa de Verdaderos Positivos', fontsize=13)
ax.set_title('Curvas ROC — Evaluacion E1b (cross-architecture)', fontsize=14)
ax.legend(fontsize=11, loc='lower right')
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])
ax.set_aspect('equal')
ax.grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
out_path = FIGURAS_DIR / 'roc_curves_e1b_superpuestas.png'
plt.savefig(str(out_path), dpi=300, bbox_inches='tight')
plt.show()
print(f'Guardado: {out_path}')

---
## Heatmap — AUC-ROC por arquitectura GAN × modelo detector

Ordenado por AUC media descendente (GANs más fáciles de detectar arriba). Usa `desglose_data`; si la sesión se reinició, recarga desde el CSV.

In [ ]:
# Recargar desde CSV si desglose_data no esta en memoria
if 'desglose_data' not in globals() or not desglose_data:
    print('desglose_data no en memoria, cargando CSV...')
    csv_path = FIGURAS_DIR / 'desglose_e1b_por_arquitectura.csv'
    desglose_data = {}
    with csv_path.open(encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            arch = row['arquitectura_gan']
            desglose_data[arch] = {
                'n_samples':    int(row['n_samples']),
                'resnet50_auc': float(row['resnet50_auc']),
                'resnet50_ap':  float(row['resnet50_ap']),
                'resnet50_acc': float(row['resnet50_acc']),
                'vit_auc':      float(row['vit_auc']),
                'vit_ap':       float(row['vit_ap']),
                'vit_acc':      float(row['vit_acc']),
                'ufd_auc':      float(row['ufd_auc']),
                'ufd_ap':       float(row['ufd_ap']),
                'ufd_acc':      float(row['ufd_acc']),
            }
    n_arqs = len(desglose_data)
    print(f'  {n_arqs} arquitecturas GAN cargadas desde {csv_path}')

modelos_hm  = ['ResNet-50', 'ViT-B/16', 'UniversalFakeDetect']
csvkeys_hm  = ['resnet50_auc', 'vit_auc', 'ufd_auc']

arqs_sorted = sorted(
    desglose_data.keys(),
    key=lambda a: -np.mean([desglose_data[a].get(k, 0.0) for k in csvkeys_hm]),
)

heat_matrix = np.array([
    [desglose_data[arch].get(k, np.nan) for k in csvkeys_hm]
    for arch in arqs_sorted
])

fig, ax = plt.subplots(figsize=(8, 10))
sns.heatmap(
    heat_matrix,
    annot=False,
    cmap='RdYlGn',
    vmin=0.5, vmax=1.0,
    xticklabels=modelos_hm,
    yticklabels=arqs_sorted,
    ax=ax,
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'AUC-ROC', 'shrink': 0.6},
)

for i, arch in enumerate(arqs_sorted):
    for j, key in enumerate(csvkeys_hm):
        val = desglose_data[arch].get(key, np.nan)
        if not np.isnan(val):
            norm = (val - 0.5) / 0.5
            text_color = 'white' if norm < 0.15 or norm > 0.85 else 'black'
            ax.text(j + 0.5, i + 0.5, f'{val:.4f}',
                    ha='center', va='center',
                    fontsize=9, fontweight='bold', color=text_color)

ax.set_title('AUC-ROC por arquitectura generativa y modelo detector', fontsize=13, pad=12)
ax.set_xlabel('Modelo detector', fontsize=12)
ax.set_ylabel('Arquitectura GAN (ordenada por AUC media)', fontsize=12)
ax.tick_params(axis='x', rotation=15, labelsize=10)
ax.tick_params(axis='y', rotation=0,  labelsize=10)

plt.tight_layout()
out_path = FIGURAS_DIR / 'heatmap_auc_por_generador.png'
plt.savefig(str(out_path), dpi=300, bbox_inches='tight')
plt.show()
print(f'Guardado: {out_path}')

---
## Guardar figuras en Drive

Copia `figuras/` completo a `MyDrive/tfm-checkpoints/figuras_memoria/`. Ejecutar al final para que las imágenes no se pierdan si cae la sesión.

In [ ]:
import shutil
from pathlib import Path

drive_fig = Path('/content/drive/MyDrive/tfm-checkpoints/figuras_memoria')
drive_fig.mkdir(parents=True, exist_ok=True)
shutil.copytree(str(FIGURAS_DIR), str(drive_fig), dirs_exist_ok=True)
print(f'Figuras copiadas a Drive: {drive_fig}')
print('Archivos guardados:')
for f in sorted(drive_fig.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:55s}  {size_kb:>7.1f} KB')